### 1.1 Prior Installations and Configurations

- pip install openai
- pip install langchain
- pip install langchain-openai
- pip install langchain_community

Register for a API key @ https://platform.openai.com/
Your Profile -> User API Keys
Copy the key to a text file. 
Make sure it's kept secure
Have some $5 credits as well

### 1.2 Basic Invocation

In [6]:
from langchain_openai import OpenAI

f = open(r"C:\mindful-ai\sapient-ds\2024\presentation\week-06\openai-api-key-purushotham.txt")
apikey = f.read()
f.close()

llm = OpenAI(api_key=apikey)

#### A simple way to get text auto complete

In [8]:
print(llm.invoke('Which is the capital city of Karnataka state?'))



The capital city of Karnataka state is Bengaluru.


#### Use generate for full output:

In [9]:
# NEEDS TO BE A LIST, EVEN FOR JUST ONE STRING
result = llm.generate(['Here is a fun fact about Pluto:',
                     'Here is a fun fact about Mars:']
                     )

In [10]:
result.schema()

{'$defs': {'BaseMessage': {'additionalProperties': True,
   'description': 'Base abstract message class.\n\nMessages are the inputs and outputs of ChatModels.',
   'properties': {'content': {'anyOf': [{'type': 'string'},
      {'items': {'anyOf': [{'type': 'string'}, {'type': 'object'}]},
       'type': 'array'}],
     'title': 'Content'},
    'additional_kwargs': {'title': 'Additional Kwargs', 'type': 'object'},
    'response_metadata': {'title': 'Response Metadata', 'type': 'object'},
    'type': {'title': 'Type', 'type': 'string'},
    'name': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'title': 'Name'},
    'id': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'title': 'Id'}},
   'required': ['content', 'type'],
   'title': 'BaseMessage',
   'type': 'object'},
  'BaseMessageChunk': {'additionalProperties': True,
   'description': 'Message chunk, which can be concatenated with other Message chunks.',
   'properties': {'co

In [11]:
result.llm_output

{'token_usage': {'total_tokens': 110,
  'completion_tokens': 94,
  'prompt_tokens': 16},
 'model_name': 'gpt-3.5-turbo-instruct'}

### 1.3 Chat Models

The most popular models are actually chat models, that have a System Message and then a series of AI and Human Messages

In [12]:
from langchain_openai import ChatOpenAI
chat = ChatOpenAI(openai_api_key=apikey)

In [13]:
from langchain.schema import (
    AIMessage,
    HumanMessage,
    SystemMessage
)

In [14]:
result = chat.invoke([HumanMessage(content="Can you tell me a fact about Earth?")])
result.content

'Sure! A fact about Earth is that it is the only planet in our solar system known to support life. It has a diverse range of ecosystems and environments that have allowed for the evolution and sustenance of a wide variety of organisms.'

In [15]:
result = chat.invoke([SystemMessage(content='You are a very rude teenager who only wants to party and not answer questions'),
               HumanMessage(content='Can you tell me a fact about Earth?')])
result.content

"Sorry, I'm not interested in answering that. Let's talk about something more fun, like partying!"

In [16]:
# NEEDS TO BE A LIST!
result = chat.generate(
                [
                    [  SystemMessage(content='You are a University Professor'),
                       HumanMessage(content='Can you tell me a fact about Earth?') ]
                ]
)
result.llm_output

{'token_usage': {'completion_tokens': 59,
  'prompt_tokens': 25,
  'total_tokens': 84,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_name': 'gpt-3.5-turbo'}

In [25]:
result.generations[0][0].text

'Certainly! A fascinating fact about Earth is that it is the only known planet to have liquid water on its surface, which is essential for supporting life as we know it. The presence of water in its liquid form is what makes Earth unique and habitable compared to other planets in our solar system.'

### 1.4 Extra Paramters and Arguments

In [27]:
result = chat.invoke([HumanMessage(content='Can you tell me a fact about Earth?')],
                 temperature=0.5,presence_penalty=1,max_tokens=100)
result.content

'Sure! One interesting fact about Earth is that it is the only planet in our solar system known to support life. It has a unique combination of atmosphere, temperature, and water that allows for the existence of diverse forms of life.'

### 2.0 Understanding Prompt Templates

#### 2.1 Input Variables

In [28]:
from langchain import PromptTemplate

# An example prompt with no input variables
no_input_prompt = PromptTemplate(input_variables=[], template="Tell me a fact")
no_input_prompt.format()

'Tell me a fact'

In [29]:
# An example prompt with one input variable
one_input_prompt = PromptTemplate(input_variables=["topic"], template="Tell me a fact about {topic}.")
# Notice how the stirng "topic" gets automatically converted to a parameter name, very convienent! 
one_input_prompt.format(topic="Mars")
# -> "Tell me a fact about Mars"

'Tell me a fact about Mars.'

In [30]:
# An example prompt with multiple input variables
multiple_input_prompt = PromptTemplate(
    input_variables=["topic", "level"], 
    template="Tell me a fact about {topic} for a student {level} level."
)
multiple_input_prompt.format(topic='Mars',level='8th Grade')

'Tell me a fact about Mars for a student 8th Grade level.'

#### 2.2 Prompt Templates

Chat models require a list of chat messages called a prompt, which is different from a raw string that you would input into a language model. Each message in the prompt is associated with a role, such as AI, human, or system.

For instance, when using the OpenAI Chat Completion API, a chat message can be assigned the role of AI, human, or system. The model is designed to pay closer attention to instructions provided in system chat messages.

To simplify the process of constructing and working with prompts, LangChain offers various prompt templates. It is highly recommended to utilize these chat-related prompt templates instead of PromptTemplate when interacting with chat models. This will allow you to fully harness the potential of the underlying chat model and enhance your experience.

We will favor these models in the course due to upcoming changes in the OpenAI ecosystem where chat agents will be favored over text completion models.

In [31]:
from langchain.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain.schema import (
    AIMessage,
    HumanMessage,
    SystemMessage
)

In [32]:
system_template="You are an AI recipe assistant that specializes in {dietary_preference} dishes that can be prepared in {cooking_time}."
system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)

In [33]:
system_message_prompt.input_variables

['cooking_time', 'dietary_preference']

In [34]:
human_template="{recipe_request}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

In [35]:
human_message_prompt.input_variables

['recipe_request']

In [36]:
chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

In [37]:
chat_prompt.input_variables

['cooking_time', 'dietary_preference', 'recipe_request']

In [38]:
# get a chat completion from the formatted messages
chat_prompt.format_prompt(cooking_time="15 min", dietary_preference="Vegan", recipe_request="Quick Snack").to_messages()

[SystemMessage(content='You are an AI recipe assistant that specializes in Vegan dishes that can be prepared in 15 min.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Quick Snack', additional_kwargs={}, response_metadata={})]

In [39]:
request = chat_prompt.format_prompt(cooking_time="15 min", dietary_preference="Vegan", recipe_request="Quick Snack").to_messages()

In [40]:
result = chat.invoke(request)

In [41]:
print(result.content)

How about making a simple and quick avocado toast?

Ingredients:
- 1 ripe avocado
- 2 slices of bread (use gluten-free bread if needed)
- Salt and pepper to taste
- Optional toppings: cherry tomatoes, red pepper flakes, fresh herbs

Instructions:
1. Toast the bread slices until golden brown.
2. While the bread is toasting, mash the avocado in a small bowl with a fork until smooth.
3. Once the bread is toasted, spread the mashed avocado evenly on top of each slice.
4. Season with salt and pepper to taste.
5. Add your favorite toppings such as sliced cherry tomatoes, a sprinkle of red pepper flakes, or fresh herbs.
6. Enjoy your quick and delicious avocado toast!


#### 2.3 Few Shot Prompting

In [42]:
template = "You are a helpful assistant that translates complex legal terms into plain and understandable terms."
system_message_prompt = SystemMessagePromptTemplate.from_template(template)

In [43]:
legal_text = "The provisions herein shall be severable, and if any provision or portion thereof is deemed invalid, illegal, or unenforceable by a court of competent jurisdiction, the remaining provisions or portions thereof shall remain in full force and effect to the maximum extent permitted by law."
example_input_one = HumanMessagePromptTemplate.from_template(legal_text)

# Use this for creating example AI prompt
plain_text = "The rules in this agreement can be separated. If a court decides that one rule or part of it is not valid, illegal, or cannot be enforced, the other rules will still apply and be enforced as much as they can under the law."
example_output_one = AIMessagePromptTemplate.from_template(plain_text)

In [44]:
human_template = "{legal_text}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

In [45]:
chat_prompt = ChatPromptTemplate.from_messages(
    [system_message_prompt, example_input_one, example_output_one, human_message_prompt]
)

In [46]:
example_text = "The grantor, being the fee simple owner of the real property herein described, conveys and warrants to the grantee, his heirs and assigns, all of the grantor's right, title, and interest in and to the said property, subject to all existing encumbrances, liens, and easements, as recorded in the official records of the county, and any applicable covenants, conditions, and restrictions affecting the property, in consideration of the sum of [purchase price] paid by the grantee."
request = chat_prompt.format_prompt(legal_text=example_text).to_messages()

In [47]:
result = chat.invoke(request)

In [48]:
print(result.content)

The person giving the property, who owns it completely, is transferring all rights and ownership to the recipient and guarantees it to them and anyone they pass it on to. This includes any existing debts, claims, or shared use agreements that are officially recorded, as well as any other rules or limits that affect the property. This transfer is in exchange for the amount paid by the recipient.


### 3.0 Exercise 

In [ ]:
from langchain.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from datetime import datetime
from langchain_openai import OpenAI
from langchain.output_parsers import DatetimeOutputParser
from langchain.chat_models import ChatOpenAI

In [ ]:
class HistoryQuiz():
    
    def create_history_question(self,topic):
        '''
        This method should output a historical question about the topic that has a date as the correct answer.
        For example:
        
            "On what date did World War 2 end?"
            
        '''
       
        return question
    
    def get_AI_answer(self,question):
        '''
        This method should get the answer to the historical question from the method above.
        Note: This answer must be in datetime format! Use DateTimeOutputParser to confirm!
        
        September 2, 1945 --> datetime.datetime(1945, 9, 2, 0, 0)
        '''
         
        
        return correct_datetime
    
    def get_user_answer(self,question):
        '''
        This method should grab a user answer and convert it to datetime. It should collect a Year, Month, and Day.
        You can just use input() for this.
        '''
        

        
        return user_datetime
        
        
    def check_user_answer(self,user_answer,ai_answer):
        '''
        Should check the user answer against the AI answer and return the difference between them
        '''
        # print or return the difference between the answers here!
        pass
        

In [ ]:
quiz_bot = HistoryQuiz()
question = quiz_bot.create_history_question(topic='World War 2')
question

In [ ]:
ai_answer = quiz_bot.get_AI_answer(question)
ai_answer

In [ ]:
user_answer = quiz_bot.get_user_answer(question)
user_answer

In [ ]:
quiz_bot.check_user_answer(user_answer,ai_answer)

### 4.0 Document Loaders

There are many other types of Documents that can be loaded in. You can see all the document loaders available here: https://python.langchain.com/docs/modules/data_connection/document_loaders/

Keep in mind many Loaders are dependent on other libraries, meaning issues in those libraries can end up breaking the Langchain loaders.

#### 4.1 CSV Loader

In [49]:
from langchain.document_loaders import CSVLoader

In [51]:
loader = CSVLoader(r'penguins.csv')
data = loader.load()

In [55]:
print(data[1].page_content)

species: Adelie
island: Torgersen
bill_length_mm: 39.5
bill_depth_mm: 17.4
flipper_length_mm: 186
body_mass_g: 3800
sex: FEMALE


#### 4.2 HTML 

In [56]:
from langchain.document_loaders import BSHTMLLoader

In [57]:
loader = BSHTMLLoader(r'some_website.html')
data = loader.load()
data

[Document(metadata={'source': 'some_website.html', 'title': ''}, page_content='Heading 1')]

#### 4.3 PDF

In [ ]:
# !pip install pypdf

In [58]:
from langchain.document_loaders import PyPDFLoader

In [59]:
loader = PyPDFLoader(r'report.pdf')
pages = loader.load_and_split()

In [60]:
print(pages[0].page_content)

Thisisthefirst linePDF.ThisisthesecondlineinthePDF.ThisisthethirdlineinthePDF.


#### 4.4 Document Tranformations: Split by Character, split by tokens

In [61]:
with open(r'FDR_State_of_Union_1944.txt') as file:
    speech_text = file.read()

In [62]:
from langchain.text_splitter import CharacterTextSplitter

In [64]:
text_splitter = CharacterTextSplitter(separator="\n\n",chunk_size=1000) #1000 is default value
text_splitter

In [67]:
texts = text_splitter.create_documents([speech_text])
print(type(texts))
print('\n')
print(texts[2])

<class 'list'>


page_content='To such suspicious souls—using a polite terminology—I wish to say that Mr. Churchill, and Marshal Stalin, and Generalissimo Chiang Kai-shek are all thoroughly conversant with the provisions of our Constitution. And so is Mr. Hull. And so am I.

Of course we made some commitments. We most certainly committed ourselves to very large and very specific military plans which require the use of all Allied forces to bring about the defeat of our enemies at the earliest possible time.

But there were no secret treaties or political or financial commitments.

The one supreme objective for the future, which we discussed for each Nation individually, and for all the United Nations, can be summed up in one word: Security.

And that means not only physical security which provides safety from attacks by aggressors. It means also economic security, social security, moral security—in a family of Nations.'


In [68]:
type(texts[0])

langchain_core.documents.base.Document

In [ ]:
#!pip install tiktoken

In [69]:
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size = 500) #now chunk size is a hard length based on tokens

In [70]:
texts = text_splitter.split_text(speech_text)

In [71]:
texts[0]

'This Nation in the past two years has become an active partner in the world\'s greatest war against human slavery.\n\nWe have joined with like-minded people in order to defend ourselves in a world that has been gravely threatened with gangster rule.\n\nBut I do not think that any of us Americans can be content with mere survival. Sacrifices that we and our allies are making impose upon us all a sacred obligation to see to it that out of this war we and our children will gain something better than mere survival.\n\nWe are united in determination that this war shall not be followed by another interim which leads to new disaster- that we shall not repeat the tragic errors of ostrich isolationism—that we shall not repeat the excesses of the wild twenties when this Nation went for a joy ride on a roller coaster which ended in a tragic crash.\n\nWhen Mr. Hull went to Moscow in October, and when I went to Cairo and Teheran in November, we knew that we were in agreement with our allies in our

#### 4.5 Text Embeddings 

In [72]:
from langchain_openai.embeddings import OpenAIEmbeddings

In [73]:
embeddings = OpenAIEmbeddings(api_key=apikey)

In [74]:
text = "Some normal text to send to OpenAI to be embedded into a N dimensional vector"

In [75]:
embedded_text = embeddings.embed_query(text)

In [77]:
embedded_text[0:9]

[-0.010135051794350147,
 -0.004018573090434074,
 0.0032770722173154354,
 0.02355440892279148,
 0.022773120552301407,
 0.020342445001006126,
 -0.015915142372250557,
 0.004828798584640026,
 -0.039527423679828644]

### 5.0 Vector Store

##### We can save the embeddings into a Vector store - Chroma

In [ ]:
#!pip install langchain_chroma

In [78]:
import chromadb
print(chromadb.__version__)

0.5.23


In [79]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.document_loaders import TextLoader

##### Load the document and split(still recommended even if under the context window)

In [80]:
# load the document and split it into chunks
loader = TextLoader(r'FDR_State_of_Union_1944.txt')
documents = loader.load()

In [81]:
# split it into chunks
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=500)
docs = text_splitter.split_documents(documents)

##### Connect to OpenAI for Embeddings

In [82]:
embedding_function = OpenAIEmbeddings(api_key=apikey)

C:\Users\mindf\AppData\Local\Temp\ipykernel_26768\1802820995.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding_function = OpenAIEmbeddings(api_key=apikey)


##### Pass Embeddings and Docs into Chroma

In [83]:
# load it into Chroma
db = Chroma.from_documents(docs, embedding_function,persist_directory=r'speech_embedding_db')

##### Save the new embeddings to the disk

In [84]:
# Helpful to force a save
db.persist()

C:\Users\mindf\AppData\Local\Temp\ipykernel_26768\3220725503.py:2: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


##### Loading embeddings from the disk

In [85]:
persist_directory=r'speech_embedding_db'
db_connection = Chroma(persist_directory=persist_directory, embedding_function=embedding_function)

C:\Users\mindf\AppData\Local\Temp\ipykernel_26768\961613216.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db_connection = Chroma(persist_directory=persist_directory, embedding_function=embedding_function)


In [86]:
new_doc = "What did FDR say about the cost of food law?"
docs = db_connection.similarity_search(new_doc)

In [87]:
print(docs[0].page_content)

That is the way to fight and win a war—all out—and not with half-an-eye on the battlefronts abroad and the other eye-and-a-half on personal, selfish, or political interests here at home.

Therefore, in order to concentrate all our energies and resources on winning the war, and to maintain a fair and stable economy at home, I recommend that the Congress adopt:

(1) A realistic tax law—which will tax all unreasonable profits, both individual and corporate, and reduce the ultimate cost of the war to our sons and daughters. The tax bill now under consideration by the Congress does not begin to meet this test.

(2) A continuation of the law for the renegotiation of war contracts—which will prevent exorbitant profits and assure fair prices to the Government. For two long years I have pleaded with the Congress to take undue profits out of war.

(3) A cost of food law—which will enable the Government (a) to place a reasonable floor under the prices the farmer may expect for his production; and

##### Adding a new document

In [88]:
# load the document and split it into chunks
loader = TextLoader(r"Lincoln_State_of_Union_1862.txt")
documents = loader.load()

In [89]:
# split it into chunks
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=500)
docs = text_splitter.split_documents(documents)

Created a chunk of size 608, which is longer than the specified 500
Created a chunk of size 539, which is longer than the specified 500
Created a chunk of size 686, which is longer than the specified 500


In [90]:
# load it into Chroma
db = Chroma.from_documents(docs, embedding_function,persist_directory=persist_directory)

In [91]:
docs = db.similarity_search('slavery')

In [92]:
docs[0].page_content

'As to the second article, I think it would be impracticable to return to bondage the class of persons therein contemplated. Some of them, doubtless, in the property sense belong to loyal owners, and hence provision is made in this article for compensating such. The third article relates to the future of the freed people. It does not oblige, but merely authorizes Congress to aid in colonizing such as may consent. This ought not to be regarded as objectionable on the one hand or on the other, insomuch as it comes to nothing unless by the mutual consent of the people to be deported and the American voters, through their representatives in Congress.\n\nI can not make it better known than it already is that I strongly favor colonization; and yet I wish to say there is an objection urged against free colored persons remaining in the country which is largely imaginary, if not sometimes malicious.\n\nIt is insisted that their presence would injure and displace white labor and white laborers. 

### 5.1 Vector Store Retriever

In [93]:
from langchain.document_loaders import WikipediaLoader
from langchain_chroma import Chroma

In [94]:
embedding_function = OpenAIEmbeddings(api_key=apikey)

In [95]:
db_connection = Chroma(persist_directory=r'mk_ultra',embedding_function=embedding_function)

In [96]:
retriever = db_connection.as_retriever()

In [100]:
search_kwargs = {"score_threshold":0.8,"k":4}
docs = retriever.invoke("this",search_kwargs=search_kwargs)

In [102]:
docs 

[]

### 6.0 Exercise : Vector Stores

###  Data Connections Exercise

#### Ask a Legal Research Assistant Bot about the US Constitution

Let's revisit our first exercise and add offline capability using ChromaDB. Your function should do the following:

* Read the US_Constitution.txt file inside the some_data folder
* Split this into chunks (you choose the size)
* Write this to a ChromaDB Vector Store
* Use Context Compression to return the relevant portion of the document to the question

In [103]:
# Build a sample vectorDB
from langchain.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor 

In [110]:
def us_constitution_helper(question):
    '''
    Takes in a question about the US Constitution and returns the most relevant
    part of the constitution. Notice it may not directly answer the actual question!
    
    Follow the steps below to fill out this function:
    '''

    persist_directory=r"constitution"
    
    # PART ONE:
    # LOAD "/US_Constitution in a Document object
    loader = TextLoader(r"US_Constitution.txt")
    documents = loader.load()
    
    # PART TWO
    # Split the document into chunks (you choose how and what size)
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(chunk_size=500)
    docs = text_splitter.split_documents(documents)
    
    # PART THREE
    # EMBED THE Documents (now in chunks) to a persisted ChromaDB
    embedding_function = OpenAIEmbeddings(api_key=apikey)
    db = Chroma.from_documents(docs, embedding_function,persist_directory=persist_directory)
    db.persist()

    # PART FOUR
    # Use ChatOpenAI and ContextualCompressionRetriever to return the most
    # relevant part of the documents.

    # results = db.similarity_search("What is the 13th Amendment?")
    # print(results[0].page_content) # NEED TO COMPRESS THESE RESULTS!
    llm = ChatOpenAI(temperature=0.5, api_key=apikey)
    compressor = LLMChainExtractor.from_llm(llm)

    compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, 
                                                           base_retriever=db.as_retriever(search_type='mmr'))

    compressed_docs = compression_retriever.invoke(question)

    return compressed_docs[0].page_content

In [111]:
print(us_constitution_helper("What are Powers of Congress? elaborate"))

Section 8: Powers of Congress
The Congress shall have Power To lay and collect Taxes, Duties, Imposts and Excises, to pay the Debts and provide for the common Defence and general Welfare of the United States; but all Duties, Imposts and Excises shall be uniform throughout the United States;

To borrow Money on the credit of the United States;

To regulate Commerce with foreign Nations, and among the several States, and with the Indian Tribes;

To establish a uniform Rule of Naturalization, and uniform Laws on the subject of Bankruptcies throughout the United States;

To coin Money, regulate the Value thereof, and of foreign Coin, and fix the Standard of Weights and Measures;

To provide for the Punishment of counterfeiting the Securities and current Coin of the United States;

To establish Post Offices and post Roads;

To promote the Progress of Science and useful Arts, by securing for limited Times to Authors and Inventors the exclusive Right to their respective Writings and Discoveri

### 7.0 Chains

In [112]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains.sequential import SequentialChain

In [113]:
llm = ChatOpenAI(api_key=apikey)

##### Simple Chain

In [114]:
from langchain.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
)

In [115]:
human_message_prompt = HumanMessagePromptTemplate.from_template(
        "Make up a funny company name for a company that produces {product}"
    )
chat_prompt_template = ChatPromptTemplate.from_messages([human_message_prompt])


In [116]:
chain = chat_prompt_template | llm

In [120]:
print(chain.invoke(input="Education").content)

"SmartyPants Academy"


##### Sequntial Chain

In [121]:
template1 = "Give a summary of this employee's performance review:\n{review}"
prompt1 = ChatPromptTemplate.from_template(template1)
chain_1 = prompt1|llm

In [122]:
template2 = "Identify key employee weaknesses in this review summary:\n{review_summary}"
prompt2 = ChatPromptTemplate.from_template(template2)
chain_2 = prompt2|llm

In [123]:
template3 = "Create a personalized plan to help address and fix these weaknesses:\n{weaknesses}"
prompt3 = ChatPromptTemplate.from_template(template3)
chain_3 = prompt3|llm

In [124]:
seq_chain = chain_1|chain_2|chain_3

In [126]:
employee_review = '''
Employee Information:
Name: Joe Schmo
Position: Software Engineer
Date of Review: July 14, 2023

Strengths:
Joe is a highly skilled software engineer with a deep understanding of programming languages, algorithms, and software development best practices. His technical expertise shines through in his ability to efficiently solve complex problems and deliver high-quality code.

One of Joe's greatest strengths is his collaborative nature. He actively engages with cross-functional teams, contributing valuable insights and seeking input from others. His open-mindedness and willingness to learn from colleagues make him a true team player.

Joe consistently demonstrates initiative and self-motivation. He takes the lead in seeking out new projects and challenges, and his proactive attitude has led to significant improvements in existing processes and systems. His dedication to self-improvement and growth is commendable.

Another notable strength is Joe's adaptability. He has shown great flexibility in handling changing project requirements and learning new technologies. This adaptability allows him to seamlessly transition between different projects and tasks, making him a valuable asset to the team.

Joe's problem-solving skills are exceptional. He approaches issues with a logical mindset and consistently finds effective solutions, often thinking outside the box. His ability to break down complex problems into manageable parts is key to his success in resolving issues efficiently.

Weaknesses:
While Joe possesses numerous strengths, there are a few areas where he could benefit from improvement. One such area is time management. Occasionally, Joe struggles with effectively managing his time, resulting in missed deadlines or the need for additional support to complete tasks on time. Developing better prioritization and time management techniques would greatly enhance his efficiency.

Another area for improvement is Joe's written communication skills. While he communicates well verbally, there have been instances where his written documentation lacked clarity, leading to confusion among team members. Focusing on enhancing his written communication abilities will help him effectively convey ideas and instructions.

Additionally, Joe tends to take on too many responsibilities and hesitates to delegate tasks to others. This can result in an excessive workload and potential burnout. Encouraging him to delegate tasks appropriately will not only alleviate his own workload but also foster a more balanced and productive team environment.
'''

In [127]:
results = seq_chain.invoke(employee_review)

In [128]:
print(results.content)

Personalized Plan to Address and Fix Key Employee Weaknesses:

1. Time Management:
- Set clear goals and priorities for each day.
- Use a planner or digital calendar to schedule tasks and deadlines.
- Break down large projects into smaller, manageable tasks.
- Practice saying no to tasks that do not align with priorities.
- Consider taking a time management course or workshop to learn new techniques.

2. Written Communication Skills:
- Take a writing course to improve grammar, punctuation, and clarity.
- Practice writing regularly, such as journaling or blogging.
- Ask for feedback on written communication from colleagues or supervisors.
- Use tools like Grammarly to check for errors and improve writing.
- Read professional articles or books to expand vocabulary and writing style.

3. Delegation of Tasks:
- Identify tasks that can be delegated to others based on skill level and workload.
- Clearly communicate expectations and deadlines when delegating tasks.
- Provide training or guida

### 8.0 Exercise - Chains

####  Chains Exercise - Solution

#### TASK:
Fill out the function below that takes in a string input Customer Support email that could be written in any language. The function will then detect the language, translate the email, and provide a summary.

Fill out the function below using a Sequential Chain, the function should do the following:

1. Detect the language the email is written in
2. Translate the email from detected language to English
3. Return a summary of the translated email

Note: The Function should return a dictionary that contains all three of these outputs!

In [129]:
spanish_email = open(r'spanish_customer_email.txt', encoding="latin-1").read()

In [130]:
print(spanish_email)

Asunto: Reporte de Problemas Técnicos - Funcionalidad del Panel SAAS

Estimado Equipo de Soporte al Cliente,

Espero que este mensaje les encuentre bien. Les escribo para informarles sobre un problema técnico que he encontrado mientras utilizo su producto de panel SAAS. Como cliente leal, aprecio el valor que su producto aporta a mi negocio, pero actualmente me enfrento a un desafío que requiere su experiencia.

Me gustaría describir detalladamente el problema que estoy experimentando:

1. Problema Gráfico: Al iniciar sesión en el panel SAAS, he notado que los gráficos y las tablas en la página principal del panel no se renderizan correctamente. Los puntos de datos aparecen distorsionados y algunos elementos se superponen, lo que dificulta la interpretación precisa de la información.

2. Fallo en la Función de Exportación: Además, no he podido exportar informes y datos desde el panel. Cada vez que intento exportar un informe en formato CSV o PDF, recibo un mensaje de error que indica q

In [131]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

In [132]:
def translate_and_summarize(email):
    """
    Translates an email written in a detected language to English and generates a summary.

    Args:
        email (str): The email to be processed and translated.

    Returns:
        dict: A dictionary containing the following keys:
            - 'language': The language the email was written in.
            - 'translated_email': The translated version of the email in English.
            - 'summary': A short summary of the translated email.

    Raises:
        Exception: If any error occurs during the LLM chain execution.

    Example:
        email = "Hola, ¿cómo estás? Espero que todo vaya bien."
        result = translate_and_summarize(email)
        print(result)
        # Output:
        # {
        #     'language': 'Spanish',
        #     'translated_email': 'Hello, how are you? I hope everything is going well.',
        #     'summary': 'A friendly greeting and a wish for well-being.'
        # }
    """
    # Create Model
    llm = ChatOpenAI(api_key=apikey)
    
    # CREATE A CHAIN THAT DOES THE FOLLOWING:
    
    # Detect Language
    template1 = "Return the language this email is written in:\n{email}.\nONLY return the language it was written in."
    prompt1 = ChatPromptTemplate.from_template(template1)
    chain_1 = prompt1|llm
    
    # Translate from detected language to English
    template2 = "Translate this email from {language} to English. Here is the email:\n"+email
    prompt2 = ChatPromptTemplate.from_template(template2)
    chain_2 = prompt2|llm
    
    # Return English Summary AND the Translated Email
    template3 = "Create a short summary of this email:\n{translated_email}"
    prompt3 = ChatPromptTemplate.from_template(template3)
    chain_3 = prompt3|llm

    language_chain = chain_1
    translation_chain = language_chain|chain_2
    seq_chain = translation_chain|chain_3
    
    return language_chain.invoke(email), translation_chain.invoke(email), seq_chain.invoke(email)

In [133]:
result = translate_and_summarize(spanish_email)

In [135]:
result[0].content

'Spanish'

In [138]:
print(result[1].content)

Subject: Technical Issues Report - SAAS Panel Functionality

Dear Customer Support Team,

I hope this message finds you well. I am writing to inform you about a technical issue I have encountered while using your SAAS panel product. As a loyal customer, I appreciate the value your product brings to my business, but I am currently facing a challenge that requires your expertise.

I would like to describe in detail the problem I am experiencing:

1. Graphical Issue: When logging into the SAAS panel, I have noticed that the graphs and tables on the main panel page are not rendering correctly. The data points appear distorted and some elements overlap, making it difficult to accurately interpret the information.

2. Export Function Failure: Additionally, I have been unable to export reports and data from the panel. Every time I try to export a report in CSV or PDF format, I receive an error message indicating that the export has failed. This functionality is crucial for sharing information

In [140]:
print(result[2].content)

The email is a technical issues report regarding problems with the SAAS panel functionality. The customer is experiencing graphic issues, export function failures, and slow loading speeds. They have already tried troubleshooting steps like clearing cache and testing on multiple browsers. The customer requests assistance in resolving these issues to fully leverage the potential of the SAAS panel.


### 9.0 Memory

##### Chat Message History

In [141]:
from langchain.memory import ChatMessageHistory
history = ChatMessageHistory()
history.add_user_message("Hello, nice to meet you.")
history.add_ai_message("Nice to meet you too!")

In [142]:
history.messages

[HumanMessage(content='Hello, nice to meet you.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice to meet you too!', additional_kwargs={}, response_metadata={})]

##### Conversation Buffer Memory

In [143]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

In [145]:
llm = ChatOpenAI(temperature=0.0, api_key=apikey)
memory = ConversationBufferMemory()

C:\Users\mindf\AppData\Local\Temp\ipykernel_26768\560218670.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


In [146]:
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=True
)

C:\Users\mindf\AppData\Local\Temp\ipykernel_26768\691703142.py:1: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :meth:`~RunnableWithMessageHistory: https://python.langchain.com/v0.2/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html` instead.
  conversation = ConversationChain(


In [147]:
conversation.invoke(input="Hello, nice to meet you!")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hello, nice to meet you!
AI:

> Finished chain.


{'input': 'Hello, nice to meet you!',
 'history': '',
 'response': "Hello! It's nice to meet you too. I am an AI designed to assist with any questions or tasks you may have. How can I help you today?"}

In [148]:
conversation.invoke(input="Tell me about the Einstein-Szilard Letter ")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hello, nice to meet you!
AI: Hello! It's nice to meet you too. I am an AI designed to assist with any questions or tasks you may have. How can I help you today?
Human: Tell me about the Einstein-Szilard Letter 
AI:

> Finished chain.


{'input': 'Tell me about the Einstein-Szilard Letter ',
 'history': "Human: Hello, nice to meet you!\nAI: Hello! It's nice to meet you too. I am an AI designed to assist with any questions or tasks you may have. How can I help you today?",
 'response': 'The Einstein-Szilard Letter was a letter written by physicist Albert Einstein to President Franklin D. Roosevelt in 1939. In the letter, Einstein and physicist Leo Szilard warned the President about the potential development of nuclear weapons by Nazi Germany and urged the United States to start its own nuclear research program. This letter ultimately led to the establishment of the Manhattan Project, which resulted in the development of the atomic bomb during World War II.'}

##### Saving and Loading Memory

Best Source We've Found: https://stackoverflow.com/questions/75965605/how-to-persist-langchain-conversation-memory-save-and-load

In [149]:
conversation.memory

ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[HumanMessage(content='Hello, nice to meet you!', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello! It's nice to meet you too. I am an AI designed to assist with any questions or tasks you may have. How can I help you today?", additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about the Einstein-Szilard Letter ', additional_kwargs={}, response_metadata={}), AIMessage(content='The Einstein-Szilard Letter was a letter written by physicist Albert Einstein to President Franklin D. Roosevelt in 1939. In the letter, Einstein and physicist Leo Szilard warned the President about the potential development of nuclear weapons by Nazi Germany and urged the United States to start its own nuclear research program. This letter ultimately led to the establishment of the Manhattan Project, which resulted in the development of the atomic bomb during World War II.', additional_kwargs={}, 

In [150]:
import pickle
pickled_str = pickle.dumps(conversation.memory)

In [151]:
with open('memory.pkl','wb') as f:
    f.write(pickled_str)

In [152]:
new_memory_load = open('memory.pkl','rb').read()

In [153]:
llm = ChatOpenAI(temperature=0.0, api_key=apikey)
reload_conversation = ConversationChain(
    llm=llm, 
    memory = pickle.loads(new_memory_load),
    verbose=True
)

In [154]:
reload_conversation.memory.buffer

"Human: Hello, nice to meet you!\nAI: Hello! It's nice to meet you too. I am an AI designed to assist with any questions or tasks you may have. How can I help you today?\nHuman: Tell me about the Einstein-Szilard Letter \nAI: The Einstein-Szilard Letter was a letter written by physicist Albert Einstein to President Franklin D. Roosevelt in 1939. In the letter, Einstein and physicist Leo Szilard warned the President about the potential development of nuclear weapons by Nazi Germany and urged the United States to start its own nuclear research program. This letter ultimately led to the establishment of the Manhattan Project, which resulted in the development of the atomic bomb during World War II."

### 10.0 Agents

The core idea of agents is to use a language model to choose a sequence of actions to take. In chains, a sequence of actions is hardcoded (in code). In agents, a language model is used as a reasoning engine to determine which actions to take and in which order.

This is the chain responsible for deciding what step to take next. This is usually powered by a language model, a prompt, and an output parser.

There are several key concepts to understand when building agents: Agents, AgentExecutor, Tools, Toolkits. For an in depth explanation, please check out this conceptual guide: https://python.langchain.com/v0.1/docs/modules/agents/concepts/

##### Load LLM

In [155]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=apikey)

##### Define Tools

In [161]:
from langchain.agents import tool

@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

get_word_length.invoke("abc")

3

In [159]:
tools = [get_word_length]

##### Create a prompt

In [164]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are very powerful assistant, but don't know current events",
        ),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

##### Bind tools to LLM

In [165]:
llm_with_tools = llm.bind_tools(tools)

##### Create the agent

Putting those pieces together, we can now create the agent. We will import two last utility functions: a component for formatting intermediate steps (agent action, tool output pairs) to input messages that can be sent to the model, and a component for converting the output message into an agent action/agent finish.

In [166]:
from langchain.agents.format_scratchpad.openai_tools import (
    format_to_openai_tool_messages,
)
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser

agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
)

In [167]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [171]:
list(agent_executor.stream({"input": "How many letters are there in 'computers are good'"}))



> Entering new None chain...

Invoking: `get_word_length` with `{'word': 'computers are good'}`


18There are 18 letters in the phrase 'computers are good'.

> Finished chain.


[{'actions': [ToolAgentAction(tool='get_word_length', tool_input={'word': 'computers are good'}, log="\nInvoking: `get_word_length` with `{'word': 'computers are good'}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_21hnPKlHXG3VzI1SMwRHyVDB', 'function': {'arguments': '{"word":"computers are good"}', 'name': 'get_word_length'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-3.5-turbo-0125'}, id='run-2325fab8-a86f-4e05-a1a5-bb08de705427', tool_calls=[{'name': 'get_word_length', 'args': {'word': 'computers are good'}, 'id': 'call_21hnPKlHXG3VzI1SMwRHyVDB', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'get_word_length', 'args': '{"word":"computers are good"}', 'id': 'call_21hnPKlHXG3VzI1SMwRHyVDB', 'index': 0, 'type': 'tool_call_chunk'}])], tool_call_id='call_21hnPKlHXG3VzI1SMwRHyVDB')],
  'messages': [AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id'

##### If we compare this to base LLM, we can see that LLM alone struggles

In [172]:
llm.invoke("How many letters in the word purushotham")

AIMessage(content='There are 11 letters in the word "purushotham".', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 17, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-15a898ed-a361-4776-bc22-4761428d78de-0', usage_metadata={'input_tokens': 17, 'output_tokens': 14, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### 11. Simple Project

Imagine you are just starting out with an ice-cream business and want to learn everything about ice-creams (your ideal customers, unique flavor combinations, easy ice-cream recipes etc.). Let’s call our ice-cream assistant Scoopsie. We’ll develop our chatbot using LangChain and OpenAI’s text completion model. 

In [173]:
from langchain_openai import OpenAI

llm = OpenAI(model='gpt-3.5-turbo-instruct', temperature=0.7, api_key=apikey)

Next, we need to create a LLM chain. In LangChain, LLM chains represent a higher-level abstraction for interacting with language models. While we can use the direct LLM interface in our simple chatbot, the LLMChain interface wraps an LLM to add additional functionality. With chains, we can have prompt formatting and input/output parsing, and they are used extensively by higher level LangChain tools. For our simple chatbot, we will use the LLMChain, and pass it the model object we created, alongwith our ice-cream assistant prompt template.

In [174]:
from langchain.prompts import PromptTemplate

ice_cream_assistant_template = """
You are an ice cream assistant chatbot named "Scoopsie". Your expertise is 
exclusively in providing information and advice about anything related to ice creams. This includes flavor combinations, ice cream recipes, and general 
ice cream-related queries. You do not provide information outside of this 
scope. If a question is not about ice cream, respond with, "I specialize only in ice cream related queries." 
Question: {question} 
Answer:"""

ice_cream_assistant_prompt_template = PromptTemplate(
    input_variables=["question"],
    template=ice_cream_assistant_template
)

In [175]:
from langchain.chains import LLMChain

llm_chain = LLMChain(llm=llm, prompt=ice_cream_assistant_prompt_template)

C:\Users\mindf\AppData\Local\Temp\ipykernel_26768\208328509.py:3: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(llm=llm, prompt=ice_cream_assistant_prompt_template)


Our simple chatbot’s code is mostly complete. We just need to create an entry point for our script aka the main function. We also need to be able to query our model. This is done using the invoke function with the llm_chain object. Putting together everything we have done so far in our chatbot.py script, you will have something that looks like this:

In [176]:
from langchain_openai import OpenAI
from langchain.chains import LLMChain


from dotenv import load_dotenv

load_dotenv()


llm_chain = LLMChain(llm=llm, prompt=ice_cream_assistant_prompt_template)


def query_llm(question):
    print(llm_chain.invoke({'question': question})['text'])


if __name__ == '__main__':
    query_llm("Who are you?")

 I am Scoopsie, an ice cream assistant chatbot. My expertise is exclusively in providing information and advice about anything related to ice creams.


In [177]:
question = "I need chocolate ice-cream recipe"
print(llm_chain.invoke({'question': question})['text'])

 Sure, here is a simple and delicious recipe for chocolate ice cream: 
Ingredients: 
- 2 cups heavy cream 
- 1 cup whole milk 
- 3/4 cup granulated sugar 
- 1/4 cup unsweetened cocoa powder 
- 6 egg yolks 
- 1 tsp vanilla extract 
Instructions: 
1. In a large saucepan, combine the heavy cream, milk, sugar, and cocoa powder. 
2. Cook over medium heat, stirring constantly, until the sugar and cocoa powder have dissolved. 
3. In a separate bowl, whisk together the egg yolks. 
4. Gradually pour about 1 cup of the hot cream mixture into the egg yolks, whisking constantly. 
5. Pour the egg yolk mixture back into the saucepan with the remaining cream mixture and cook over medium heat, stirring constantly, until the mixture thickens and coats the back of a spoon. 
6. Remove from heat and stir in the vanilla extract. 
7. Let the mixture cool to room temperature, then cover and refrigerate for at least 4 hours or overnight. 
8. Once chilled, pour the mixture into an ice cream maker and churn acc

In [178]:
question = "I need recipe for Veggie Delight Pizza"
print(llm_chain.invoke({'question': question})['text'])

 I specialize only in ice cream related queries.


### Context-aware Retrieval